# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This notebook keeps the project architecture fixed before model evaluation: **classification + regression + ranking**.

The five model features were already selected in Assignment 4 using March-only information and are not reopened here:

1. `aggregate_ctr`
2. `median_position`
3. `position_slope_per_day`
4. `position_iqr`
5. `content_age_days`

### Classification — Logistic Regression

The binary target is future decline: `future_impression_change < 0`.

A scaled **Logistic Regression** is the first learned classifier because the target is binary, the model produces an interpretable probability needed by the final ranking, and it provides a deliberately simple comparison against the frozen training-prior baseline. No class weighting or test-driven tuning is used.

Primary metric: **ROC-AUC**.

Frozen Assignment 5 baseline: **ROC-AUC = 0.500**.

### Regression — Random Forest Regressor

The continuous target is signed `future_impression_change`.

A deliberately moderate **Random Forest Regressor** is used because the five March features may relate to future movement nonlinearly and through interactions. The configuration is fixed before held-out evaluation:

- `n_estimators=300`
- `max_depth=6`
- `min_samples_leaf=10`
- `random_state=42`
- `n_jobs=-1`

No hyperparameter search is performed against the held-out clients.

Primary metric: **RMSE**.

Frozen Assignment 5 baseline: **RMSE = 1.4311**.

### Ranking — transparent risk × severity score

The learned ranking combines the two model outputs rather than creating a new manual ranking label:

`predicted_decline_severity = max(0, -predicted_future_change)`

`ranking_score = p_decline × predicted_decline_severity`

A larger score therefore means the page is both more likely to decline and predicted to deteriorate more severely. This is a transparent prioritisation score, not a causal-effect estimate.

Primary metric: **Precision@50**.

Frozen Assignment 5 ranking baseline: **Precision@50 = 0.480**.

The six held-out clients remain sealed for model selection. The feature set, model families, hyperparameters, ranking formula, split and primary metrics are fixed before held-out model performance is inspected.

In [ ]:
# STEP 1 — reconstruct the locked March feature frame and load frozen baselines.
# This cell does NOT inspect held-out model performance or tune any method.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# Hugging Face token stays private.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Reconstruct the exact Assignment-4/5 modeling population.
# April is used here only for the already-locked >=20-day outcome-observability rule;
# no April outcome value is used for feature choice or model selection.
march_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)

eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(
        ["client_hash_id", "exposure_tier"],
        observed=False,
        group_keys=False,
    )
    .head(40)
    .reset_index(drop=True)
)

balanced_keys = balanced_poc[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("balanced_keys", balanced_keys)

# Construct only the five already-locked March-safe model features.
march_features = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            DATE_DIFF(
                'day',
                DATE '2026-03-01',
                f.report_date
            )::DOUBLE AS day_index,
            f.gsc_impressions::DOUBLE AS impressions,
            f.gsc_clicks::DOUBLE AS clicks,
            CASE
                WHEN f.gsc_avg_position >= 1
                THEN f.gsc_avg_position::DOUBLE
                ELSE NULL
            END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k
            USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
        MEDIAN(valid_position) AS median_position,
        REGR_SLOPE(valid_position, day_index)
            FILTER (WHERE valid_position IS NOT NULL)
            AS position_slope_per_day,
        (
            QUANTILE_CONT(valid_position, 0.75)
            - QUANTILE_CONT(valid_position, 0.25)
        ) AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-31'
        )::DOUBLE AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k
        USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr",
    "median_position",
    "position_slope_per_day",
    "position_iqr",
    "content_age_days",
]

feature_frame = march_features.merge(
    age_feature,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Load Assignment-5 receipts rather than redefining the benchmark.
output_dir = Path("../outputs")
with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)

with open(output_dir / "multitask_baseline_benchmark.json", "r", encoding="utf-8") as fh:
    frozen_benchmarks = json.load(fh)

# Locked learned-method specification. These choices precede held-out evaluation.
MODEL_SPEC = {
    "classification": {
        "model": "StandardScaler + LogisticRegression",
        "primary_metric": "ROC-AUC",
    },
    "regression": {
        "model": "RandomForestRegressor",
        "n_estimators": 300,
        "max_depth": 6,
        "min_samples_leaf": 10,
        "random_state": 42,
        "n_jobs": -1,
        "primary_metric": "RMSE",
    },
    "ranking": {
        "formula": "p_decline * max(0, -predicted_future_change)",
        "k": 50,
        "primary_metric": "Precision@50",
    },
}

# Contract assertions: fail loudly if prior state has drifted.
assert len(feature_frame) == 2520
assert feature_frame["client_hash_id"].nunique() == 21
assert feature_frame[FINAL_FEATURES].notna().all().all()
assert np.isfinite(feature_frame[FINAL_FEATURES].to_numpy(dtype=float)).all()
assert split_manifest["random_state"] == 42
assert split_manifest["test_size"] == 0.25
assert split_manifest["train_pages"] == 1800
assert split_manifest["test_pages"] == 720
assert len(split_manifest["client_overlap"]) == 0
assert np.isclose(
    frozen_benchmarks["classification"]["roc_auc"], 0.5
)
assert np.isclose(
    frozen_benchmarks["regression"]["rmse"], 1.4311128557344202
)
assert np.isclose(
    frozen_benchmarks["ranking"]["precision_at_50"], 0.48
)

print("ASSIGNMENT 6 — METHOD CONTRACT")
print("Feature rows:", len(feature_frame))
print("Clients:", feature_frame["client_hash_id"].nunique())
print("Features:", FINAL_FEATURES)
print("Train pages:", split_manifest["train_pages"])
print("Test pages:", split_manifest["test_pages"])
print("Client overlap:", len(split_manifest["client_overlap"]))
print("\nFrozen baselines:")
print(
    "Classification ROC-AUC:",
    frozen_benchmarks["classification"]["roc_auc"],
)
print(
    "Regression RMSE:",
    frozen_benchmarks["regression"]["rmse"],
)
print(
    "Ranking Precision@50:",
    frozen_benchmarks["ranking"]["precision_at_50"],
)
print("\nLocked learned methods:")
for task, spec in MODEL_SPEC.items():
    print(task, "->", spec)

display(feature_frame.head(10))


## 2. Split design

Assignment 6 inherits the **exact frozen Assignment 5 validation split** rather than creating a new one.

The split was originally created with:

`GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)`

grouped by `client_hash_id`.

This gives:

- **15 training clients / 1,800 pages**
- **6 held-out clients / 720 pages**
- **0 clients shared between train and test**

The grouped design is required because pages from the same client can share site-level behaviour. A random page split could therefore make the test set artificially easy by allowing the same client to appear on both sides.

The feature window remains **1–31 March 2026**. The future outcome remains **1–30 April 2026**. Every page must have at least 20 usable GSC days in both months.

The three targets/evaluation roles are unchanged:

- **Classification:** `future_decline = 1` when `future_impression_change < 0`.
- **Regression:** continuous signed `future_impression_change`.
- **Ranking relevance:** the same binary future-decline outcome, evaluated at `K = 50`.

This split intentionally preserves the observed client shift found in Assignment 5 rather than hiding it. Training decline prevalence is substantially higher than held-out prevalence, and the mean future change also shifts between train and test. That makes the benchmark harder but more honest: Assignment 6 is testing whether the learned models generalise to unseen clients.

In [ ]:
# STEP 2 — reconstruct the locked future targets and apply the exact frozen client split.
# No model is fitted in this cell.

# Future outcome for the exact locked 2,520-page population.
con.register(
    "model_keys",
    feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates()
)

march_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS march_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS april_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

target_frame = march_target.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"]
    - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]

target_frame["future_decline"] = (
    target_frame["future_impression_change"] < 0
).astype(int)

modeling_frame = feature_frame.merge(
    target_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "march_usable_days",
            "april_usable_days",
            "future_impression_change",
            "future_decline",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Reuse the exact pseudonymized client lists frozen in Assignment 5.
train_clients = set(split_manifest["train_clients"])
test_clients = set(split_manifest["test_clients"])

train_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(train_clients)
].copy()

test_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(test_clients)
].copy()

# Contract checks.
assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert (modeling_frame["march_usable_days"] >= 20).all()
assert (modeling_frame["april_usable_days"] >= 20).all()
assert (modeling_frame["march_avg_impressions_per_day"] > 0).all() if "march_avg_impressions_per_day" in modeling_frame.columns else True

assert len(train_frame) == split_manifest["train_pages"] == 1800
assert len(test_frame) == split_manifest["test_pages"] == 720
assert train_frame["client_hash_id"].nunique() == 15
assert test_frame["client_hash_id"].nunique() == 6

observed_train_clients = set(train_frame["client_hash_id"].unique())
observed_test_clients = set(test_frame["client_hash_id"].unique())

assert observed_train_clients == train_clients
assert observed_test_clients == test_clients
assert observed_train_clients.isdisjoint(observed_test_clients)

assert set(train_frame.index).isdisjoint(set(test_frame.index))
assert len(train_frame) + len(test_frame) == len(modeling_frame)

# Feature/target separation checks.
for forbidden in [
    "future_impression_change",
    "future_decline",
    "march_usable_days",
    "april_usable_days",
]:
    assert forbidden not in FINAL_FEATURES

X_train = train_frame[FINAL_FEATURES].copy()
X_test = test_frame[FINAL_FEATURES].copy()

y_cls_train = train_frame["future_decline"].astype(int).copy()
y_cls_test = test_frame["future_decline"].astype(int).copy()

y_reg_train = train_frame["future_impression_change"].astype(float).copy()
y_reg_test = test_frame["future_impression_change"].astype(float).copy()

assert X_train.notna().all().all()
assert X_test.notna().all().all()
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "pages": len(train_frame),
            "clients": train_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_train.mean(),
            "mean_future_change": y_reg_train.mean(),
        },
        {
            "split": "test",
            "pages": len(test_frame),
            "clients": test_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_test.mean(),
            "mean_future_change": y_reg_test.mean(),
        },
    ]
)

print("ASSIGNMENT 6 — FROZEN SPLIT CHECK")
print("Population pages:", len(modeling_frame))
print("Population clients:", modeling_frame["client_hash_id"].nunique())
print("Train pages:", len(train_frame))
print("Train clients:", train_frame["client_hash_id"].nunique())
print("Test pages:", len(test_frame))
print("Test clients:", test_frame["client_hash_id"].nunique())
print(
    "Client overlap:",
    len(observed_train_clients.intersection(observed_test_clients)),
)
print("Feature count:", len(FINAL_FEATURES))
print("Classification target:", "future_decline")
print("Regression target:", "future_impression_change")
print("Ranking K:", split_manifest["ranking_k"])
print("\nObserved split shift:")
display(split_summary)

print("\nTrain feature frame:")
display(X_train.head())
print("\nTest feature frame:")
display(X_test.head())


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.